In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import seaborn as sns
from scipy import stats

ox.settings.use_cache = True
sns.set_style("whitegrid")

fig_dir = Path("../paper/latex/")
fp_treeness_results = "./data/treeness-results.csv"
fp_indicators = "./data/indicators-all.csv"

# Baseline Cases: Other Kinds of Networks

In [ ]:
# FIRST, inspect a few non-street networks to compare to our street networks below
# calculate link redundancy: assume m and n are for undirected graphs
# if we don't know `c`, approximate it as 1 (since its irrelevant for sufficiently large graphs)
def calculate_link_redundancy(n, m, c=1):
    cr = m - n + c
    ratio = cr / m
    return round(ratio, 3)


# msn messenger 2006 from: https://arxiv.org/abs/0803.0939
# "180 million nodes and 1.3 billion undirected edges"
print("MSN", calculate_link_redundancy(n=180_000_000, m=1_300_000_000))

# European airline network (undirected) from https://icon.colorado.edu/
# https://www.nature.com/srep/2013/130227/srep01344/full/srep01344.html
print("EAN", calculate_link_redundancy(n=450, m=3588))

# paris subway network (undirected)
# from: world subways 2009 https://icon.colorado.edu/networks
# https://royalsocietypublishing.org/doi/full/10.1098/rsif.2012.0259
print("Paris", calculate_link_redundancy(n=299, m=712))

# delhi bus route network (undirected)
# from: india bus routes (2016) https://icon.colorado.edu/networks
# https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0168478
print("Delhi", calculate_link_redundancy(n=1554, m=4285))

# Load Our Indicators + Treeness Results

In [ ]:
# load the street networks indicators
df_ind = pd.read_csv(fp_indicators)
df_ind.shape

In [ ]:
# load the treeness indicators
df_mst = pd.read_csv(fp_treeness_results)
df_mst["uc_id"] = df_mst["id"].str.split("-", expand=True)[1].astype(int)
df_mst = df_mst.drop(columns=["id"])
df_mst.shape

In [ ]:
# merge them together and only keep cities with >100 nodes
df = df_ind.merge(df_mst, on="uc_id", how="inner")
df = df[df["node_count"] > 100]
df.shape

In [ ]:
# create/transform any necessary columns
df["intersect_density"] = df["intersect_count"] / df["built_up_area_m2"]

df["length_redundancy"] = 1 - df["length_ratio"]
df["length_redundancy_cc"] = 1 - df["length_ratio_cc"]

cols = {"circuit_rank_ratio": "link_redundancy", "circuit_rank_ratio_cc": "link_redundancy_cc"}
df = df.rename(columns=cols)

# Sanity Check

In [ ]:
# edge_count_ratio + link_redundancy should equal 1, because...
# link_redundancy = (edge_count - node_count + component_count) / edge_count
# edge_count_ratio = edge_count_mst / edge_count_original = 1 - link_redundancy
np.allclose(df["edge_count_ratio"], 1 - df["link_redundancy"])

# Examine treeness: link and length redundancy

In [ ]:
# describe link redundancy for our street networks
# typical is ~28%
# compare this to the other network types analyzed at top of notebook
df_redundancy = df[["link_redundancy", "length_redundancy"]]
table = df_redundancy.describe().round(3)
print(table.to_latex(float_format="%.3f"))
table

In [ ]:
# coefficients of variation are relatively small
df_redundancy.std() / df_redundancy.mean()

In [ ]:
# no extreme outliers in terms of tukey fences at 3x IQR
def extreme_outliers(s):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 3 * iqr
    upper = q3 + 3 * iqr

    return s[(s < lower) | (s > upper)]


print(len(extreme_outliers(df_redundancy["link_redundancy"])))
print(len(extreme_outliers(df_redundancy["length_redundancy"])))

In [ ]:
# visualize distributions of link and length redundancy
fig, ax = plt.subplots(figsize=(5, 6))
bins = 40
alpha = 0.6
ax = df["length_redundancy"].hist(
    bins=bins,
    density=False,
    alpha=alpha,
    ax=ax,
    color="#003399",
    label="Length Redundancy ($\\lambda$)",
)
ax = df["link_redundancy"].hist(
    bins=bins,
    density=False,
    alpha=alpha,
    ax=ax,
    color="#cc0000",
    label="Link Redundancy ($\\theta$)",
)
ax.set_xlim(0, 1)
ax.set_ylim(0, 800)
ax.set_ylabel("Observations")
_ = plt.legend(fontsize=9)
fig.tight_layout()
fig.savefig(fig_dir / "fig_histogram_indicators.pdf", bbox_inches="tight")

In [ ]:
# look at values across world regions
df_theta = df.groupby("world_region")["link_redundancy"].agg(
    Min="min", Mean="mean", Median=np.median, Max="max"
)
df_theta.columns = pd.MultiIndex.from_product([["theta"], df_theta.columns])
df_lambda = df.groupby("world_region")["length_redundancy"].agg(
    Min="min", Mean="mean", Median=np.median, Max="max"
)
df_lambda.columns = pd.MultiIndex.from_product([["lambda"], df_lambda.columns])
df_table = pd.concat([df_theta, df_lambda], axis=1).sort_index().round(3)
print(df_table.to_latex(float_format="%.3f"))
df_table

# Statistical Associations

In [ ]:
# transform variables for interpretable axes and linear fits
df["length_total_km"] = df["length_total"] / 1000
df["intersect_density_km2"] = df["intersect_density"] * 1e6
df["built_up_area_km2"] = df["built_up_area_m2"] / 1e6
df["pop_density_km2_1000s"] = (df["resident_pop"] / 1000) / df["built_up_area_km2"]
df["gdp_ppp_percap"] = df["gdp_ppp"] * df["built_up_area_km2"] / df["resident_pop"]
df["built_up_area_km2_log"] = np.log(df["built_up_area_km2"])
df["length_total_km_log"] = np.log(df["length_total_km"])
df["gdp_ppp_percap_log"] = np.log(df["gdp_ppp_percap"])

In [ ]:
# calculate correlations
y_cols = [
    "length_total_km_log",  # log
    "intersect_density_km2",
    "orientation_entropy",
    "bc_gini",
    "k_avg",
    "elev_median",
    "circuity",
    "elev_std",
    "built_up_area_km2_log",  # log
    "pop_density_km2_1000s",
    "gdp_ppp_percap_log",
]
corrs = []
for x_col in ["link_redundancy", "length_redundancy"]:
    for y_col in y_cols:
        r, p = stats.pearsonr(df[x_col], df[y_col])
        corrs.append((x_col, y_col, r, p))

# show results
cols = ["x_col", "y_col", "r", "p"]
df_corrs = pd.DataFrame(corrs, columns=cols)  # .set_index(["x_col", "y_col"])
df_corrs.sort_values(by="r", key=abs, ascending=False).round(3)

In [ ]:
stats.pearsonr(df["link_redundancy"], df["length_redundancy"])

In [ ]:
# decriptive stats for "other" urban indicators of interest
# avg degree, BC Gini, circuity
# intersect density, orientation entropy, total length
# hilliness, population density, built up area
df[y_cols].describe().T.round(2)

In [ ]:
# visualize relationships between link+length redundancy vs other
# urban indicators of interest
fig, axes = plt.subplots(3, 3, figsize=(9, 10), facecolor="w")
x_cols = [
    "k_avg",
    "bc_gini",
    "circuity",
    "gdp_ppp_percap_log",
    "intersect_density_km2",
    "length_total_km_log",
    "elev_std",
    "pop_density_km2_1000s",
    "built_up_area_km2_log",
]
x_labels = [
    "Average Node Degree",
    "Gini of Node Betweenness Centralities",
    "Average Circuity",
    "GDP (PPP) Per Capita (log)",
    "Intersection Density",
    "Total Network Length (log)",
    "Std Dev of Node Elevations",
    "Population Density",
    "Built-up Area (log)",
]
y_cols = ["link_redundancy", "length_redundancy"]
y_labels = ["Link Redundancy ($\\theta$)", "Length Redundancy ($\\lambda$)"]
xlims = [
    (2, 3.7),
    (0.4, 0.95),
    (1, 1.11),
    (3.5, 10),
    (0, 800),
    (3.5, 8),
    (1, 60),
    (4.5, 100),
    (0, 4.5),
]
colors = ["#cc0000", "#003399"]
for x_col, ax, xlim, x_label in zip(x_cols, axes.flat, xlims, x_labels, strict=False):
    r_label = ""
    for y_col, color, y_label in zip(y_cols, colors, y_labels, strict=False):
        ax.scatter(df[x_col], df[y_col], s=1, alpha=0.03, color=color)

        m, b, r, p, se = stats.linregress(df[x_col], df[y_col])
        ax.axline((0, b), slope=m, color=color, linestyle="--", linewidth=1, label=y_label)

        indicator = "$\\lambda$" if y_col == "length_redundancy" else "$\\theta$"
        stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        r_label += f"$r$({indicator})={r:.2f}{stars}$~~~$"

    if x_col in {"k_avg", "gdp_ppp_percap_log", "elev_std"}:
        ax.set_ylabel("Redundancy")

    ax.set_title(r_label.rstrip())
    ax.set_xlabel(x_label)
    ax.set_xlim(xlim)
    ax.set_ylim(0.05, 0.65)
    ax.set_box_aspect(1)
    if x_col == "k_avg":
        ax.legend()

fig.tight_layout()
fig.savefig(fig_dir / "fig_scatter_regressions.pdf", bbox_inches="tight")

# Visualize example small street networks: tree-like vs not

In [ ]:
count = 3  # how many of each?
min_edges = 600
max_edges = 900
mask = (df["edge_count_orig_cc"] > min_edges) & (df["edge_count_orig_cc"] < max_edges)
df_subset = df[mask].sort_values("length_ratio_cc")
uc_ids = df_subset["uc_id"].head(count).tolist() + df_subset["uc_id"].tail(count).tolist()
uc_ids

In [ ]:
# %%script false --no-raise-error
bgcolor = "w"
fig, axes = plt.subplots(2, count, figsize=(count * 4, 8), facecolor=bgcolor)
for uc_id, ax in zip(uc_ids, axes.flat, strict=False):
    row = df[df["uc_id"] == uc_id].iloc[0]
    root = Path("/data/snm/models/graphml")
    fp = (
        root
        / f"{row['country']}-{row['country_iso']}"
        / f"{row['core_city']}-{row['uc_id']}.graphml"
    )
    Gu = ox.convert.to_undirected(ox.truncate.largest_component(ox.io.load_graphml(fp)))
    mst = nx.minimum_spanning_tree(Gu, weight="length")
    print(row["length_ratio"])
    ax.set_facecolor(bgcolor)
    fig, ax = ox.plot.plot_graph(
        Gu,
        node_size=0,
        ax=ax,
        edge_linewidth=1,
        bgcolor=bgcolor,
        edge_color="r",
        show=False,
        close=False,
    )
    fig, ax = ox.plot.plot_graph(
        mst,
        node_size=0,
        ax=ax,
        edge_linewidth=1.2,
        bgcolor=bgcolor,
        edge_color="k",
        show=False,
        close=False,
    )
fig.savefig(fig_dir / "fig_network_examples.pdf", bbox_inches="tight")

# Examine just the megacities

In [ ]:
df_big = df[df["resident_pop"] > 10_000_000]
df_big.shape

In [ ]:
cols = ["country", "core_city", "resident_pop", "length_redundancy", "link_redundancy"]
df_big.sort_values("link_redundancy")[cols]

In [ ]:
# example figure: grid-like buenos aires vs tree-like bangkok
dist = 900
nt = "drive"
graphs = []
points = [(-34.733, -58.282), (13.730, 100.438)]
for point in points:
    G = ox.graph.graph_from_point(point, dist=dist, network_type=nt, truncate_by_edge=True)
    graphs.append(G)

fig, axs = plt.subplots(1, 2, figsize=(10, 5), facecolor="w")

for ax, G, width in zip(axs, graphs, [3, 2], strict=False):
    ax.set_facecolor("w")
    ox.plot.plot_figure_ground(
        G, ax=ax, dist=805, bgcolor="w", color="black", default_width=width, show=False, close=False
    )

fig.tight_layout(pad=0)
fig.savefig(fig_dir / "fig_ba_vs_bangkok.pdf", bbox_inches="tight")
plt.show()

## Robustness check 1: largest connected component only

In [ ]:
# what % of the typical graph's edges and length are in its largest
# connected component?
df[["cc_edge_ratio", "cc_length_ratio"]].median()

In [ ]:
# compare link+length redundancy for full graph vs vs largest component only
cols = ["link_redundancy", "link_redundancy_cc", "length_redundancy", "length_redundancy_cc"]
df[cols].describe().round(3)

In [ ]:
# check if statistical associations remain same sign for the
# largest component as they were above for the full graph
x_cols = [
    "k_avg",
    "bc_gini",
    "circuity",
    "gdp_ppp_percap_log",
    "intersect_density_km2",
    "length_total_km_log",
    "elev_std",
    "pop_density_km2_1000s",
    "built_up_area_km2_log",
]
y_cols = ["link_redundancy_cc", "length_redundancy_cc"]

results = []
for x in x_cols:
    for y in y_cols:
        m, b, r, p, se = stats.linregress(df[x], df[y])
        results.append([x, y, r, p])

cols = ["x", "y", "r", "p"]
pd.DataFrame(results, columns=cols).set_index(["x", "y"]).sort_index().round(3)

# Robustness check 2: neatnet-cleaned graphs

In [ ]:
df_neat = pd.read_csv("data/treeness-results_neat.csv")

df_neat["uc_id"] = df_neat["id"].str.split("-", expand=True)[1].astype(int)
df_neat = df_neat.drop(columns=["id"])
df_neat = df_ind.merge(df_neat, on="uc_id", how="inner")
df_neat = df_neat[df_neat["node_count"] > 100]

df_neat["length_redundancy"] = 1 - df_neat["length_ratio"]
df_neat["length_redundancy_cc"] = 1 - df_neat["length_ratio_cc"]
cols = {"circuit_rank_ratio": "link_redundancy", "circuit_rank_ratio_cc": "link_redundancy_cc"}
df_neat = df_neat.rename(columns=cols)

df_neat["length_total_km"] = df_neat["length_total"] / 1000
df_neat["intersect_density"] = df_neat["intersect_count"] / df_neat["built_up_area_m2"]
df_neat["intersect_density_km2"] = df_neat["intersect_density"] * 1e6
df_neat["built_up_area_km2"] = df_neat["built_up_area_m2"] / 1e6
df_neat["pop_density_km2_1000s"] = (df_neat["resident_pop"] / 1000) / df_neat["built_up_area_km2"]
df_neat["gdp_ppp_percap"] = (
    df_neat["gdp_ppp"] * df_neat["built_up_area_km2"] / df_neat["resident_pop"]
)
df_neat["built_up_area_km2_log"] = np.log(df_neat["built_up_area_km2"])
df_neat["length_total_km_log"] = np.log(df_neat["length_total_km"])
df_neat["gdp_ppp_percap_log"] = np.log(df_neat["gdp_ppp_percap"])

y_cols = ["link_redundancy", "length_redundancy"]
df_neat = df_neat[y_cols + x_cols]

df_neat.shape

In [ ]:
# check if statistical associations remain same sign for the
# neatnet-cleaned graph as they were above for the full graph
results = []
for x in x_cols:
    for y in y_cols:
        m, b, r, p, se = stats.linregress(df_neat[x], df_neat[y])
        results.append([x, y, r, p])

cols = ["x", "y", "r", "p"]
pd.DataFrame(results, columns=cols).set_index(["x", "y"]).sort_index().round(3)